# PhotoMedGemma — Kaggle Validation Notebook

**Purpose:** Run MedGemma 4B-IT on each breast cancer PNG, capture layer-0 attention
activations, compare against our photonic chip simulation.

## Setup checklist
1. **Kaggle secret:** Add-ons → Secrets → `HF_TOKEN`
2. **Model access:** Accept terms at https://huggingface.co/google/medgemma-4b-it
3. **Accelerator:** Session → GPU T4 x2 (or any GPU)
4. **Dataset:** Upload 5 breast cancer PNGs as Kaggle dataset `breast-cancer-pngs`
   - Files: `10025_893612858.png`, `10011_1031443799.png`, `10042_102733848.png`,
     `10042_1648588715.png`, `10042_495770405.png`
   - Kaggle path: `/kaggle/input/breast-cancer-pngs/`

In [ ]:
!pip install -q transformers accelerate bitsandbytes Pillow numpy scipy

In [ ]:
import os, json, glob, traceback
import numpy as np
from pathlib import Path
from typing import List, Tuple, Optional

MODEL_ID      = 'google/medgemma-4b-it'
IMAGE_DIR     = '/kaggle/input/breast-cancer-pngs'
SVD_RANK      = 64
QUANTIZE_4BIT = True   # True = ~3.5 GB VRAM (T4 ok); False = ~8 GB (A100/P100)
QUERY         = 'Describe any abnormalities visible in this medical image in detail.'
OUTPUT_DIR    = Path('/kaggle/working/photomedgemma_outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

# HuggingFace token
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF token: loaded from Kaggle secrets')
except Exception:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')
    print(f'HF token: env ({bool(HF_TOKEN)})')

# Find images — check dataset path, then working dir, then synthesize
image_paths = sorted(glob.glob(f'{IMAGE_DIR}/*.png'))
if not image_paths:
    image_paths = sorted(glob.glob('/kaggle/working/*.png'))
if not image_paths:
    print('No PNGs found — generating synthetic breast phantom')
    from PIL import Image as _PIL
    _rng = np.random.default_rng(0)
    _arr = np.zeros((448, 448), np.float32) + 0.05
    _Y, _X = np.mgrid[0:448, 0:448]
    _arr += np.exp(-0.5*((((_X-224)/60)**2)+((_Y-224)/70)**2))*0.6
    _arr = (_arr + 0.02*_rng.standard_normal((448,448))).clip(0,1)
    _p = '/kaggle/working/synthetic_breast.png'
    _PIL.fromarray((_arr*255).astype(np.uint8),'L').convert('RGB').save(_p)
    image_paths = [_p]

print(f'Model:    {MODEL_ID}')
print(f'Quant:    {"4-bit NF4" if QUANTIZE_4BIT else "bfloat16"}')
print(f'Rank:     {SVD_RANK}')
print(f'Images:   {len(image_paths)}')
for p in image_paths:
    print(f'  {p}')

## 1. Load MedGemma

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

if torch.cuda.is_available():
    print(f'GPU:  {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('No GPU — running on CPU (very slow)')

processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)

if QUANTIZE_4BIT:
    bnb_cfg = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, token=HF_TOKEN,
        quantization_config=bnb_cfg,
        device_map='auto',
    )
else:
    model = AutoModelForImageTextToText.from_pretrained(
        MODEL_ID, token=HF_TOKEN,
        torch_dtype=torch.bfloat16,
        device_map='auto',
    )

model.eval()
print(f'Model class: {type(model).__name__}')
print(f'Params:      {sum(p.numel() for p in model.parameters())/1e9:.2f}B')

## 2. Auto-discover model internals
MedGemma ships as `Gemma3ForConditionalGeneration`.
Different HF versions expose layers at different paths — we probe at runtime.

In [ ]:
def _get_transformer_layers(m):
    """Return the list of transformer decoder layers, regardless of HF class."""
    probes = [
        lambda m: m.model.layers,                    # Gemma3ForConditionalGeneration
        lambda m: m.language_model.model.layers,     # PaliGemma / LLaVA
        lambda m: m.model.language_model.model.layers,
        lambda m: m.model.decoder.layers,            # OPT
        lambda m: m.transformer.h,                   # GPT-2
    ]
    for fn in probes:
        try:
            layers = fn(m)
            if layers is not None and len(layers) > 0:
                return layers
        except AttributeError:
            pass
    raise RuntimeError(
        f"Cannot find transformer layers in {type(m).__name__}.\n"
        f"Top-level children: {[n for n,_ in m.named_children()]}"
    )

def _get_attn_proj(layer, proj_name):
    """Get q/k/v/o_proj from a layer, trying self_attn and attention."""
    for attn_attr in ('self_attn', 'attention', 'attn'):
        attn = getattr(layer, attn_attr, None)
        if attn is not None:
            proj = getattr(attn, proj_name, None)
            if proj is not None:
                return proj
    raise AttributeError(f'{proj_name} not found in layer {type(layer).__name__}')

# Discover and verify
transformer_layers = _get_transformer_layers(model)
layer0 = transformer_layers[0]

print(f'Transformer layers: {len(transformer_layers)}')
print(f'Layer 0 type:       {type(layer0).__name__}')
for pn in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
    proj = _get_attn_proj(layer0, pn)
    print(f'  {pn}: {type(proj).__name__}  weight={tuple(proj.weight.shape)}')

## 3. Extract layer-0 weights (once — shared across all images)

In [ ]:
def extract_weight(layer_idx: int, proj_name: str) -> np.ndarray:
    """Extract weight as float32 numpy. Handles NF4 dequantization."""
    layer = transformer_layers[layer_idx]
    proj  = _get_attn_proj(layer, proj_name)
    w = proj.weight
    if QUANTIZE_4BIT and hasattr(w, 'quant_state'):
        try:
            import bitsandbytes as bnb
            return bnb.functional.dequantize_4bit(
                w.data, w.quant_state
            ).float().cpu().numpy()
        except Exception as e:
            print(f'  Warning: bitsandbytes dequant failed ({e}), falling back to .data')
    return w.detach().float().cpu().numpy()

W_q = extract_weight(0, 'q_proj')
W_k = extract_weight(0, 'k_proj')
W_v = extract_weight(0, 'v_proj')
W_o = extract_weight(0, 'o_proj')
WEIGHTS = {'q_proj': W_q, 'k_proj': W_k, 'v_proj': W_v, 'o_proj': W_o}

print('Layer-0 weights extracted:')
for name, W in WEIGHTS.items():
    print(f'  {name}: {W.shape}  norm={np.linalg.norm(W):.3f}')

# Save for analyze_kaggle_results.py
for name, W in [('W_q', W_q), ('W_k', W_k), ('W_v', W_v), ('W_o', W_o)]:
    np.save(OUTPUT_DIR / f'{name}_layer0.npy', W)
print('Weights saved.')

## 4. Clements photonic decomposition (inline — no local imports needed)

In [ ]:
def _mzi(theta, phi):
    c, s = np.cos(theta/2), np.sin(theta/2)
    e = np.exp(1j * phi)
    return np.array([[c, 1j/e*s], [1j*e*s, c]])

def _null(a, b):
    if abs(b) < 1e-15:
        return 0.0, 0.0
    return (2*np.arctan2(abs(b), abs(a))) % (2*np.pi), (np.angle(b)-np.angle(a)) % (2*np.pi)

def clements_decompose(U: np.ndarray, rank: int):
    """Decompose the rank×rank top-left submatrix of unitary U."""
    r = min(rank, U.shape[0])
    W = U[:r, :r].astype(complex).copy()
    mzis = []
    for col in range(r - 1):
        for row in range(r - 1, col, -1):
            th, ph = _null(W[row-1, col], W[row, col])
            W[[row-1, row], :] = _mzi(th, ph).conj().T @ W[[row-1, row], :]
            mzis.append((row-1, th, ph))
    return mzis, np.angle(np.diag(W))

def clements_fwd(mzis, ps, x: np.ndarray) -> np.ndarray:
    """Simulate Clements mesh forward pass."""
    f = x.astype(complex).copy() * np.exp(1j * ps)
    for (i, th, ph) in reversed(mzis):
        f[[i, i+1]] = _mzi(th, ph) @ f[[i, i+1]]
    return f

# Sanity-check on 4×4 identity
_m, _p = clements_decompose(np.eye(4, dtype=complex), rank=4)
_e = np.linalg.norm(clements_fwd(_m, _p, np.ones(4)) - 1)
assert _e < 1e-12, f'Clements identity error {_e:.2e}'
print(f'Clements sanity OK (err={_e:.2e})')

## 5. SVD-compile all four projections (once)

In [ ]:
from scipy.linalg import svd as scipy_svd

compiled = {}  # proj_name -> dict

for pname, W in WEIGHTS.items():
    print(f'Compiling {pname} {W.shape} ...')
    U_sv, s, Vh_sv = scipy_svd(W.astype(np.float64), full_matrices=True)
    r       = min(SVD_RANK, len(s))
    energy  = float((s[:r]**2).sum() / (s**2).sum())
    sigma_n = s[:r] / s[0]          # normalised singular values

    mzis_U,  ps_U  = clements_decompose(U_sv,  rank=r)
    mzis_Vh, ps_Vh = clements_decompose(Vh_sv, rank=r)

    # Pre-compute reference SVD output for comparison (avoids per-token SVD)
    # U_r @ diag(s[:r]) @ Vh_r  is the rank-r digital approximation
    U_r  = U_sv[:, :r]
    Vh_r = Vh_sv[:r, :]

    compiled[pname] = dict(
        mzis_U=mzis_U, ps_U=ps_U,
        mzis_Vh=mzis_Vh, ps_Vh=ps_Vh,
        sigma_n=sigma_n, s0=float(s[0]),
        U_r=U_r, Vh_r=Vh_r, s_r=s[:r],
        rank=r, energy=energy,
        n_mzis_U=len(mzis_U), n_mzis_Vh=len(mzis_Vh),
        W_shape=list(W.shape),
    )
    print(f'  rank={r}  energy={energy:.4f}  MZIs U={len(mzis_U):,} Vh={len(mzis_Vh):,}')

print('\nAll projections compiled.')

## 6. Per-image pipeline: MedGemma forward + activation capture + photonic comparison

In [ ]:
from PIL import Image

def _load_image(path: str):
    img  = Image.open(path).convert('RGB')
    stem = Path(path).stem
    return img, {'filename': Path(path).name, 'stem': stem,
                 'rows': img.height, 'columns': img.width}


def _run_medgemma(pil_img):
    """
    Run MedGemma on one image. Returns (output_ids, inputs, captured_activations, response).
    Registers / removes hooks internally.
    """
    captured = {}
    hooks    = []

    def _hook(name):
        def fn(module, inp, out):
            # out may be a tensor or tuple; always take first element
            tensor = out[0] if isinstance(out, (tuple, list)) else out
            captured[name] = {
                'input':  inp[0].detach().float().cpu().numpy(),
                'output': tensor.detach().float().cpu().numpy(),
            }
        return fn

    layer0 = transformer_layers[0]
    for pname in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
        proj = _get_attn_proj(layer0, pname)
        hooks.append(proj.register_forward_hook(_hook(pname)))

    # Two-step input building:
    # Step 1: apply_chat_template produces a plain text string (no image tensor yet)
    # Step 2: processor(text, images=[...]) produces the full input tensors
    messages = [{'role': 'user', 'content': [
        {'type': 'image'},
        {'type': 'text', 'text': QUERY},
    ]}]
    try:
        prompt = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False
        )
        inputs = processor(text=prompt, images=[pil_img], return_tensors='pt')
    except Exception:
        # Fallback for older processor versions
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True,
            tokenize=True, return_dict=True, return_tensors='pt',
        )

    # Move to same device as first model parameter (safe with device_map='auto')
    first_device = next(model.parameters()).device
    inputs = {k: v.to(first_device) if hasattr(v, 'to') else v
              for k, v in inputs.items()}

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=150, do_sample=False)

    for h in hooks:
        h.remove()

    n_in     = inputs['input_ids'].shape[1]
    response = processor.decode(output_ids[0, n_in:], skip_special_tokens=True)
    return output_ids, inputs, captured, response


def _photonic_compare(captured, stem):
    """
    Compare photonic simulation vs digital for each attention projection.

    Three error metrics are computed per projection:
      chip_fidelity_err  : photonic vs rank-r digital WITH SAME truncated input
                           → pure Clements mesh accuracy; should be ~1e-14 (machine precision)
      rank_truncation_err: rank-r SVD vs full W WITH FULL input
                           → information lost by using rank-64 instead of full rank (~0.1-0.3 is normal)
      chip_vs_full_err   : photonic vs full W WITH FULL input
                           → total error a downstream model would see

    PASS/FAIL is based on chip_fidelity_err (<1%) because that is what the photonic
    chip actually controls. rank_truncation_err is a physics constraint (chip has 64 modes,
    hidden dim is 2560) and does NOT indicate a chip defect.
    """
    activation_info   = []
    layer_comparisons = {}

    for pname in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
        if pname not in captured:
            print(f'  WARNING: no activation captured for {pname}')
            continue

        act_in  = captured[pname]['input']
        act_out = captured[pname]['output']

        # Remove batch dim if present
        if act_in.ndim  == 3: act_in  = act_in[0]
        if act_out.ndim == 3: act_out = act_out[0]

        # Save raw activations as .npy
        safe = pname.replace('_', '')
        np.save(OUTPUT_DIR / f'input_layer0_{safe}_{stem}.npy',  act_in)
        np.save(OUTPUT_DIR / f'output_layer0_{safe}_{stem}.npy', act_out)
        activation_info.append({
            'proj': pname,
            'input_shape':  list(act_in.shape),
            'output_shape': list(act_out.shape),
        })

        comp = compiled[pname]
        W64  = WEIGHTS[pname].astype(np.float64)
        r    = comp['rank']
        U_r, Vh_r, s_r = comp['U_r'], comp['Vh_r'], comp['s_r']

        n_test = min(32, act_in.shape[0])
        errs_chip, errs_trunc, errs_full = [], [], []

        for tok in range(n_test):
            x    = act_in[tok].astype(np.float64)
            x_in = x[:W64.shape[1]]          # full hidden state (e.g. 2560-dim)

            # --- Digital references ---
            y_dig      = W64 @ x_in                           # full rank, full input
            y_svd_full = (U_r * s_r) @ (Vh_r @ x_in)         # rank-r, full input

            # --- Photonic chip (64 input modes) ---
            x_r = x_in[:r].astype(complex)
            if len(x_r) < r:
                _tmp = np.zeros(r, dtype=complex); _tmp[:len(x_r)] = x_r; x_r = _tmp

            after_Vh  = clements_fwd(comp['mzis_Vh'], comp['ps_Vh'], x_r)
            after_sig = after_Vh * comp['sigma_n'] * comp['s0']
            y_u_r     = clements_fwd(comp['mzis_U'],  comp['ps_U'],  after_sig)

            y_phot        = np.zeros(W64.shape[0]); y_phot[:r] = y_u_r.real

            # Digital with SAME truncated input → isolates Clements accuracy
            # Vh_r[:, :r] = top-left r×r block of Vh; U_r[:r, :] = top-left r×r block of U
            y_svd_trunc   = (U_r[:r, :] * s_r) @ (Vh_r[:, :r] @ x_r.real)

            nd  = np.linalg.norm(y_dig)       + 1e-15
            nst = np.linalg.norm(y_svd_trunc) + 1e-15

            errs_chip.append(float(np.linalg.norm(y_phot[:r]  - y_svd_trunc) / nst))
            errs_trunc.append(float(np.linalg.norm(y_svd_full - y_dig)       / nd))
            errs_full.append(float(np.linalg.norm(y_phot      - y_dig)       / nd))

        mean_chip  = float(np.mean(errs_chip))   # pure Clements accuracy
        mean_trunc = float(np.mean(errs_trunc))  # rank-64 truncation cost
        mean_full  = float(np.mean(errs_full))   # total vs full MedGemma
        passed     = mean_chip < 0.01

        layer_comparisons[pname] = {
            'n_tokens':                 n_test,
            'rank':                     r,
            'energy':                   comp['energy'],
            'n_mzis_total':             comp['n_mzis_U'] + comp['n_mzis_Vh'],
            'mean_chip_fidelity_err':   mean_chip,
            'mean_rank_truncation_err': mean_trunc,
            'mean_chip_vs_full_err':    mean_full,
            # backward-compat keys for analyze_kaggle_results.py
            'mean_svd_vs_digital':      mean_trunc,
            'mean_photonic_vs_svd':     mean_chip,
            'pass_photonic':            passed,
        }
        status = 'PASS' if passed else 'FAIL'
        print(
            f'  {pname}: chip={mean_chip:.2e}  rank_trunc={mean_trunc:.2e}'
            f'  vs_full={mean_full:.2e}  [{status}]'
        )
    return activation_info, layer_comparisons


def run_image(image_path: str) -> dict:
    """Full pipeline for one breast cancer PNG."""
    pil_img, meta = _load_image(image_path)
    stem = meta['stem']
    print(f'\n{"="*60}')
    print(f'Image: {meta["filename"]}  ({meta["rows"]}x{meta["columns"]})')

    _, inputs, captured, response = _run_medgemma(pil_img)
    print(f'Tokens: {inputs["input_ids"].shape[1]}')
    print(f'Response: {response[:120]}...')
    print(f'Captured: {list(captured.keys())}')

    act_info, lc = _photonic_compare(captured, stem)
    all_pass = all(v['pass_photonic'] for v in lc.values())

    result = {
        'image':             meta['filename'],
        'stem':              stem,
        'image_size':        [meta['rows'], meta['columns']],
        'query':             QUERY,
        'response':          response,
        'n_input_tokens':    int(inputs['input_ids'].shape[1]),
        'activations':       act_info,
        'layer_comparisons': lc,
        'all_pass':          all_pass,
    }
    (OUTPUT_DIR / f'activations_{stem}.json').write_text(json.dumps(result, indent=2))
    print(f'  Saved activations_{stem}.json   all_pass={all_pass}')
    return result

print('Pipeline functions defined.')
print()
print('Error metric key:')
print('  chip_fidelity_err   = photonic vs rank-r digital (same input) -- pure Clements accuracy')
print('  rank_truncation_err = rank-r SVD vs full W (full input)       -- cost of 64-mode chip')
print('  chip_vs_full_err    = photonic vs full W (full input)         -- total error vs MedGemma')
print('  PASS/FAIL tests chip_fidelity_err < 1% (not truncation error)')

In [ ]:
all_results = []

for img_path in image_paths:
    try:
        all_results.append(run_image(img_path))
    except Exception as e:
        print(f'ERROR on {img_path}: {e}')
        traceback.print_exc()

print(f'\nCompleted {len(all_results)}/{len(image_paths)} images.')

## 7. Save combined `kaggle_comparison.json`
Compatible with local `scripts/analyze_kaggle_results.py`.

In [ ]:
def _mean_field(results, proj, field):
    vals = [r['layer_comparisons'].get(proj, {}).get(field)
            for r in results if r['layer_comparisons'].get(proj)]
    vals = [v for v in vals if v is not None]
    return float(np.mean(vals)) if vals else 0.0

q = compiled['q_proj']
combined = {
    'model_id':    MODEL_ID,
    'quantized':   QUANTIZE_4BIT,
    'query':       QUERY,
    'svd_rank':    SVD_RANK,
    'n_images':    len(all_results),
    'images':      [r['image'] for r in all_results],
    'per_image':   all_results,
    # legacy single-image fields expected by analyze_kaggle_results.py
    'response':    all_results[0]['response'] if all_results else '',
    'dicom_meta':  {'modality': 'PNG',
                    'patient_id': all_results[0]['stem'] if all_results else 'unknown'},
    'layer_0_q_proj': {
        'weight_shape':         q['W_shape'],
        'effective_rank':       q['rank'],
        'energy_retained':      q['energy'],
        'n_mzis_U':             q['n_mzis_U'],
        'n_mzis_Vh':            q['n_mzis_Vh'],
        'mean_svd_vs_digital':  _mean_field(all_results, 'q_proj', 'mean_svd_vs_digital'),
        'mean_photonic_vs_svd': _mean_field(all_results, 'q_proj', 'mean_photonic_vs_svd'),
        'token_comparisons':    [],
    },
}

(OUTPUT_DIR / 'kaggle_comparison.json').write_text(json.dumps(combined, indent=2))
print(f'Saved kaggle_comparison.json')

## 8. Summary

In [ ]:
SEP = '=' * 80
print(SEP)
print('PhotoMedGemma x MedGemma -- Breast Cancer Validation')
print(SEP)
print(f'Model: {MODEL_ID}   Quant: {"4-bit NF4" if QUANTIZE_4BIT else "bfloat16"}   Rank: {SVD_RANK}')
print()
print(f'{"Image":<28} {"proj":<7} {"chip_fidelity":>14} {"rank_trunc":>11} {"vs_full":>9}  Pass')
print('-' * 78)
for r in all_results:
    lc = r.get('layer_comparisons', {})
    first = True
    for pn in ('q_proj', 'k_proj', 'v_proj', 'o_proj'):
        c = lc.get(pn, {})
        chip  = c.get('mean_chip_fidelity_err',   float('nan'))
        trunc = c.get('mean_rank_truncation_err',  float('nan'))
        full  = c.get('mean_chip_vs_full_err',     float('nan'))
        ok    = c.get('pass_photonic', False)
        img_label = r['stem'][:27] if first else ''
        print(f'  {img_label:<27} {pn:<7} {chip:>14.2e} {trunc:>11.2e} {full:>9.2e}  {"YES" if ok else "NO"}')
        first = False

print()
print('Legend:')
print('  chip_fidelity : photonic vs rank-r digital, same 64-mode input  (~1e-14 = machine precision)')
print('  rank_trunc    : rank-64 SVD vs full W, full 2560-mode input     (0.1-0.3 = normal for 64/2560)')
print('  vs_full       : photonic chip vs full MedGemma                  (total system error)')
print('  PASS tests chip_fidelity < 1% — the rank_trunc loss is a chip design choice, not a defect.')

print()
print('MedGemma responses:')
print('-' * 80)
for r in all_results:
    print(f"\n[{r['stem']}]")
    for line in r['response'].split('\n')[:5]:
        print(f'  {line}')

print()
print('Output files (download from Kaggle output tab):')
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f'  {f.name:<56} {f.stat().st_size/1024:>7.0f} KB')

print()
print('Run locally after downloading:')
print('  mkdir -p output/simulations')
print('  cp ~/Downloads/kaggle_comparison.json output/simulations/')
print('  cp ~/Downloads/W_*_layer0.npy         output/simulations/')
print('  python3 scripts/analyze_kaggle_results.py \\')
print('    --results output/simulations/kaggle_comparison.json \\')
print('    --weights-dir output/simulations/')
print(SEP)